# BT4222 | JDSearch | Workstream 2: Query and Item Features

**Purpose:** Turn each search–candidate pair into a set of query-matching and product-metadata features that the downstream ranking models can use.

**Input files (from the shared JDSearch folder):** `user_subset.parquet` (85,864 search sessions) and `product_meta_data.txt` (12,141,247 product catalogue rows).

**Main outputs:** `query_item_train.parquet`, `query_item_val.parquet`, `query_item_test.parquet`, `bm25_config.json`, `feature_dictionary.csv`, and `handoff_manifest.json`.

### How to use this notebook

- **First full execution or recomputation:** Run Sections **0–6** top-to-bottom in Google Colab. Keep `MAX_SESSIONS = None` for the complete sampled dataset. The catalogue scan is large; do not repeat it unnecessarily.
- **Already completed the full run:** Sections **7–9** work from saved exported files. Mount Google Drive and execute these sections independently; you do **not** need to rerun Sections 1–6.
- **For other teammates:** Start at Section **8 (Handoff)** and Section **9 (Summary)** to see exactly what has been produced and how to use it. The actual scores are in the three Parquet files, not in this explanation.

**Scope:** This notebook engineer features, not the final ranking model. BM25 is computed as a *feature* here; Workstream 3 evaluates it as a standalone ranking baseline. Workstream 4 trains a learning-to-rank model and tests whether our features improve ranking.

**Methodological assumptions:** The full product metadata catalogue is treated as static and available at scoring time. Metadata-derived statistics do not use candidate interaction labels. Stored candidate position is a join key only and is never passed to the model as a predictor.


## 0. Configuration and dependencies

Use Colab with High-RAM runtime if available. The metadata file is ~4.5 GB and is streamed rather than loaded into pandas. For faster I/O, optionally copy input files to Colab's local disk if space permits. Do **not** put multi-GB files directly in GitHub.

Expected source folder in Drive shortcut: `/content/drive/MyDrive/JDsearch/`. Update `DATA_DIR` if necessary. Use the same `user_subset.parquet` across all six workstreams; do not resample.

In [ ]:
%pip -q install "pandas>=2.0" "pyarrow>=14" "duckdb>=1.0" "tqdm>=4.65"

In [ ]:
from pathlib import Path
from collections import Counter
from functools import lru_cache
from tqdm.auto import tqdm
import csv, json, math, gc, time, hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

DATA_DIR = Path('/content/drive/MyDrive/JDsearch')
# Keep this location consistent with your Google Drive shortcut.
OUT_DIR = DATA_DIR / 'workstream2_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Intermediate and diagnostics subfolders are created in Section 0 after RUN_TAG is known.
USER_PATH = DATA_DIR / 'user_subset.parquet'
META_PATH = DATA_DIR / 'product_meta_data.txt'

# Set to 200 for an initial fast end-to-end smoke test
MAX_SESSIONS = None
CANDIDATE_BATCH = 300
METADATA_BATCH = 30_000
FEATURE_BATCH = 20_000
FORCE_REBUILD = False

assert USER_PATH.exists(), f'Not found: {USER_PATH}'
assert META_PATH.exists(), f'Not found: {META_PATH}'
print('Inputs found. Test mode:', MAX_SESSIONS is not None)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inputs found. Test mode: False


**Execution policy:** First ran with `MAX_SESSIONS = 200` as a smoke test. Set it to `None` for the final run and rerun from the top. Full and sample outputs go to different subfolders so they cannot accidentally be mixed. The first full run can take substantial time because it streams a 4.5 GB Drive file and scores 7.67 million pairs.

In [ ]:
RUN_TAG = 'full' if MAX_SESSIONS is None else f'smoke_{MAX_SESSIONS}'
RUN_DIR = OUT_DIR / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR = RUN_DIR / 'intermediate'
DIAGNOSTICS_DIR = RUN_DIR / 'diagnostics'
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
BASE_PATH = INTERMEDIATE_DIR / 'candidate_pairs.parquet'
MATCH_PATH = INTERMEDIATE_DIR / 'matched_metadata.parquet'
FEATURE_PATH = INTERMEDIATE_DIR / 'query_item_all.parquet'
STATS_PATH = INTERMEDIATE_DIR / 'catalogue_stats.json'
BM25_PATH = RUN_DIR / 'bm25_config.json'

raw = pd.read_parquet(USER_PATH)
expected_cols = {'query','candidate_wid_list','candidate_label_list','split'}
assert expected_cols.issubset(raw.columns), expected_cols - set(raw.columns)
if MAX_SESSIONS is None:
    assert len(raw) == 85_864, ('Unexpected source row count', len(raw))
else:
    # For smoke tests sample from each split but keep ORIGINAL row IDs.
    raw = pd.concat([
        group.head(MAX_SESSIONS // 3 + 1)
        for _, group in raw.groupby('split', sort=False)
    ]).head(MAX_SESSIONS)
raw['session_id'] = raw.index.astype('int64')
assert raw['session_id'].is_unique
print('Sessions:',len(raw), '\nSplit counts:\n',raw['split'].value_counts())

Sessions: 85864 
Split counts:
 split
train    60104
val      12880
test     12880
Name: count, dtype: int64


## 1. From search sessions to candidate-level rows

The input `user_subset.parquet` has **one row per user's final search**, not one row per candidate product. Each row contains an anonymised query, a list of product IDs, an equally long list of labels, and a fixed `train`/`val`/`test` assignment.

The code below parses the encoded strings and expands them to **one row per query–candidate pair**, preserving the original row index as `session_id` and the within-session list index as `candidate_pos`. This matters because ranking is carried out *within each search session*.

- `query` contains term IDs separated by `\x18`.
- `candidate_wid_list` and `candidate_label_list` are separated by `_`.
- Labels are **0 = no recorded interaction, 1 = click, 2 = cart, 3 = purchase**.
- We preserve all label-0 candidate rows as weak negatives and do not infer negatives for products outside the candidate list.

**Expected full-run checks:** 85,864 sessions, 7,665,446 candidate rows, and 10,428 purchases. These numbers are from the group's interim report; actual output checks run below.


In [ ]:
def parse_ids(value, delimiter='_'):
    return [int(x) for x in str(value).split(delimiter) if x != '']

def parse_labels(value):
    values = [float(x) for x in str(value).split('_') if x != '']
    assert all(v in (0.0,1.0,2.0,3.0) for v in values)
    return [int(v) for v in values]

# Small, deterministic parsing checks
assert parse_ids('10_20_30') == [10,20,30]
assert parse_labels('2.0_0.0') == [2,0]
assert '60257419' in str(raw.iloc[0]['query']) or len(raw)>0

candidate_ids = set()
writer = None
try:
    for start in tqdm(range(0, len(raw), CANDIDATE_BATCH), desc='Candidate batches'):
        rows=[]
        for sid, row in raw.iloc[start:start+CANDIDATE_BATCH].iterrows():
            ids = parse_ids(row['candidate_wid_list'])
            labels = parse_labels(row['candidate_label_list'])
            assert len(ids) == len(labels), f'List mismatch in session {sid}'
            candidate_ids.update(ids)
            for j, (wid, label) in enumerate(zip(ids, labels)):
                rows.append((int(sid),j,str(row['split']),str(row['query']),wid,label))
        chunk = pd.DataFrame(rows, columns=['session_id','candidate_pos','split','query','candidate_id','label'])
        chunk = chunk.astype({'session_id':'int64','candidate_pos':'int16','candidate_id':'int64','label':'int8'})
        table = pa.Table.from_pandas(chunk,preserve_index=False)
        if writer is None: writer = pq.ParquetWriter(BASE_PATH, table.schema, compression='zstd')
        writer.write_table(table)
finally:
    if writer is not None: writer.close()
assert BASE_PATH.exists()
print('Unique candidate products:',len(candidate_ids))
print('Candidate rows:', pq.read_metadata(BASE_PATH).num_rows)
if MAX_SESSIONS is None:
    assert pq.read_metadata(BASE_PATH).num_rows == 7_665_446

Candidate batches:   0%|          | 0/287 [00:00<?, ?it/s]

Unique candidate products: 4859472
Candidate rows: 7665446


## 2. Extract product information and calculate catalogue statistics

The original session data provides product IDs but not the product text or attributes needed for matching. We therefore scan the 13-column `product_meta_data.txt` file **in chunks**, without loading the entire ~4.5 GB file into pandas.

During the *same full-catalogue scan*, the code:

1. Saves product metadata **only for the candidate IDs** in this sampled dataset. Product IDs are joined on `candidate_id = wid` in Section 3.
2. Counts how many catalogue products share a brand, shop, or level-4 category. These are **catalogue breadth**, not purchase or click popularity.
3. Computes product-name term document frequencies and average name length. These are used to calculate BM25 consistently across all splits.

Full-catalogue counts must be computed **before** reducing the dataset to candidates; otherwise they would count sampled products rather than the full catalogue. This stage reads the catalogue but never uses the labels to derive features.


In [ ]:
HEADER = ['wid','name','brand_id','brand_name',
          'cate_id_1','cate_name_1','cate_id_2','cate_name_2',
          'cate_id_3','cate_name_3','cate_id_4','cate_name_4','shop_id']

@lru_cache(maxsize=100_000)
def tokenize(s):
    if s is None or s == '' or pd.isna(s): return ()
    return tuple(x for x in str(s).split('\x18') if x != '')

# Keep all IDs as strings for joins or counters; convert wid to integer for joins.
brand_counts, shop_counts, cate4_counts = Counter(), Counter(), Counter()
doc_freq = Counter()
num_docs, total_doc_len, malformed = 0, 0, 0
writer = None
buffer = []
start_time = time.monotonic()

try:
    with open(META_PATH,'r',encoding='utf-8',newline='') as f:
        reader=csv.reader(f,delimiter='\t')
        header=next(reader)
        assert header == HEADER, f'Unexpected columns: {header}'
        for line_no, fields in enumerate(tqdm(reader,desc='Reading catalogue',unit='products'),start=2):
            if len(fields)!=13:
                malformed+=1
                continue
            wid_str,name,brand_id,brand_name,cat1,catn1,cat2,catn2,cat3,catn3,cat4,catn4,shop_id=fields
            try: wid=int(wid_str)
            except ValueError:
                malformed+=1
                continue
            name_tokens=tokenize(name)
            num_docs += 1
            total_doc_len += len(name_tokens)
            doc_freq.update(set(name_tokens))
            if brand_name: brand_counts[brand_id] += 1
            if shop_id not in ('','-1'): shop_counts[shop_id] += 1
            if cat4: cate4_counts[cat4] += 1
            if wid in candidate_ids:
                buffer.append((wid,name,brand_id,brand_name,catn1,catn2,catn3,catn4,cat4,shop_id))
            if len(buffer)>=METADATA_BATCH:
                part=pd.DataFrame(buffer,columns=['wid','name','brand_id','brand_name',
                          'cate_name_1','cate_name_2','cate_name_3','cate_name_4','cate_id_4','shop_id'])
                table=pa.Table.from_pandas(part,preserve_index=False)
                if writer is None: writer=pq.ParquetWriter(MATCH_PATH,table.schema,compression='zstd')
                writer.write_table(table)
                buffer.clear()
    if buffer:
        part=pd.DataFrame(buffer,columns=['wid','name','brand_id','brand_name',
                      'cate_name_1','cate_name_2','cate_name_3','cate_name_4','cate_id_4','shop_id'])
        table=pa.Table.from_pandas(part,preserve_index=False)
        if writer is None: writer=pq.ParquetWriter(MATCH_PATH,table.schema,compression='zstd')
        writer.write_table(table)
finally:
    if writer is not None: writer.close()

assert MATCH_PATH.exists(), 'No matching metadata rows; check source IDs.'
assert num_docs > 0 and total_doc_len > 0
stats_summary={'num_docs':num_docs,'avg_doc_len':total_doc_len/num_docs,
               'matched_unique_rows':pq.read_metadata(MATCH_PATH).num_rows,
               'candidate_unique':len(candidate_ids),'malformed_rows':malformed,
               'catalogue_scan_seconds':round(time.monotonic()-start_time,1)}
print(json.dumps(stats_summary,indent=2))
assert num_docs >= len(candidate_ids)*0.5, 'Unexpectedly small catalogue'

Reading catalogue: 0products [00:00, ?products/s]

{
  "num_docs": 12141247,
  "avg_doc_len": 27.913257180255044,
  "matched_unique_rows": 4829298,
  "candidate_unique": 4859472,
  "malformed_rows": 0,
  "catalogue_scan_seconds": 768.3
}


**Important:** Do not rerun only the catalogue cell and expect downstream variables to persist after disconnecting from Colab. The statistics below are saved to disk. This notebook presently runs the catalogue scan on each full, clean run for reproducibility; it does not assume old intermediate files are valid.

In [ ]:
# Save IDF data and metadata group-count mappings in portable compressed Parquet files.
# This is intentionally full-catalogue metadata, independent of labels/splits.
for mapping, path, key_name, val_name in [
    (doc_freq,INTERMEDIATE_DIR/'term_document_frequencies.parquet','term_id','df'),
    (brand_counts,INTERMEDIATE_DIR/'brand_catalogue_counts.parquet','brand_id','count'),
    (shop_counts,INTERMEDIATE_DIR/'shop_catalogue_counts.parquet','shop_id','count'),
    (cate4_counts,INTERMEDIATE_DIR/'cate4_catalogue_counts.parquet','cate_id_4','count')]:
    pd.DataFrame({key_name:list(mapping.keys()),val_name:list(mapping.values())}).to_parquet(
        path,index=False,compression='zstd')
with open(STATS_PATH,'w') as f: json.dump(stats_summary,f,indent=2)

# Use a fixed, explicit Lucene-style smoothed IDF variant of Okapi BM25.
# This differs slightly from the floor-for-negative-IDF behavior of rank_bm25.BM25Okapi.
BM25_K1 = 1.5
BM25_B = 0.75
bm25_config={
  'variant':'Okapi BM25 with log1p positive-smoothed Robertson IDF (Lucene style)',
  'idf_formula':'ln(1 + (N - df + 0.5)/(df + 0.5))',
  'score_formula':'sum_idf * tf*(k1+1)/(tf + k1*(1-b+b*dl/avgdl))',
  'k1':BM25_K1,'b':BM25_B,'corpus':'all valid product_meta_data.txt product names',
  'num_docs':num_docs,'avgdl':total_doc_len/num_docs,
  'token_separator':'\u0018','uses_labels':False,
  'metadata_assumption':'static complete product catalogue is available at inference',
}
with open(BM25_PATH,'w') as f: json.dump(bm25_config,f,indent=2)
print('Saved BM25 configuration:',BM25_PATH)

Saved BM25 configuration: /content/drive/MyDrive/JDsearch/workstream2_outputs/full/bm25_config.json


## 3. Join candidate rows to product metadata

We use a **left join**: each candidate row survives even if a metadata record is missing. The original number of query–candidate pairs must remain unchanged.

For missing products, feature values are null and `has_metadata = 0`. A product with available metadata but no matching query terms instead receives an actual overlap of zero. This distinction matters to the downstream model.

DuckDB streams this large join rather than building a second full table of Python objects in memory. A duplicate-ID check prevents the join from silently multiplying rows.


In [ ]:
con=duckdb.connect(database=':memory:')
con.execute("SET memory_limit='5GB'")
con.execute("SET threads=2")
# Colab local storage is faster for temporary join files.
TEMP_DIR=Path('/content/jdsearch_duckdb_tmp') if IN_COLAB else (RUN_DIR/'duckdb_tmp')
TEMP_DIR.mkdir(parents=True,exist_ok=True)
con.execute(f"SET temp_directory='{str(TEMP_DIR).replace(chr(39), chr(39)*2)}'")

# 0 duplicates should exist in the catalogue. Explicit check avoids accidental row multiplication.
check=con.execute('SELECT count(*) n, count(DISTINCT wid) u FROM read_parquet(?)',[str(MATCH_PATH)]).fetchone()
assert check[0]==check[1], f'Duplicate product metadata IDs: {check}'

join_sql="""
SELECT c.session_id,c.candidate_pos,c.split,c.query,c.candidate_id,c.label,
       m.wid AS matched_wid,m.name,m.brand_id,m.brand_name,
       m.cate_name_1,m.cate_name_2,m.cate_name_3,m.cate_name_4,
       m.cate_id_4,m.shop_id
FROM read_parquet(?) c
LEFT JOIN read_parquet(?) m ON c.candidate_id=m.wid
"""
con.execute(join_sql,[str(BASE_PATH),str(MATCH_PATH)])
reader=con.fetch_record_batch(rows_per_batch=FEATURE_BATCH)
print('Join ready for streaming feature construction')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Join ready for streaming feature construction


## 4. Compute the 11 planned features (+ one metadata-availability flag)

### Four query-to-product matching features

| Feature | Meaning | Why it might help |
|---|---|---|
| `query_name_overlap` | Fraction of **unique query terms** also in the product name | Direct textual alignment with the current query |
| `query_brand_overlap` | Fraction of unique query terms in the brand name | Detects brand-specific searches; null for missing brand |
| `query_cate_overlap` | Fraction of unique query terms in the **union of the four category names** | Detects product-category intent |
| `bm25_query_name` | BM25 score between the query and product name | Rewards informative, uncommon matching terms and accounts for name length and term frequency |

### Seven product attributes

| Feature | Meaning | Why it might help |
|---|---|---|
| `name_length` | Product-name token count | Controls for how much product description is available |
| `has_brand` | 1 if brand name is present | Separates known brands from missing brand information |
| `shop_is_missing` | 1 if shop ID is absent/-1 | Flags incomplete seller metadata |
| `cate_depth` | 3 if level-3 and level-4 category names are identical, otherwise 4 | Approximate taxonomy specificity |
| `brand_product_count` | Full-catalogue number of products for this brand | Brand catalogue breadth, **not** behavioural popularity |
| `shop_product_count` | Full-catalogue number of products for this shop | Shop catalogue breadth |
| `cate4_product_count` | Full-catalogue number of products for this category | Category catalogue breadth |

**Additional flag:** `has_metadata` indicates whether the catalogue contains the candidate product. This preserves missingness information and is not one of the 11 planned features.

### Which BM25 are we using?

This pipeline uses **Okapi BM25 with Lucene-style positive-smoothed IDF**, **`k1 = 1.5`** and **`b = 0.75`**, computed using **all valid product names in the static catalogue**. This differs from `rank_bm25.BM25Okapi`'s treatment of negative IDF; do not interchange the scores without recomputing all splits.

- **Corpus:** each catalogue product name is a document; `N` is the number of product names.
- **Document frequency:** number of product names containing a given term.
- **IDF:** `ln(1 + (N - df + 0.5)/(df + 0.5))`.
- **Scoring:** BM25 term-frequency saturation and document-length normalisation; repeated query terms contribute repeatedly, whereas overlap uses unique query terms.
- **Unit:** a dimensionless relevance score, **not** a probability and with no universal maximum.
- **Comparison:** meaningful only when corpus statistics and parameters are consistent.

The code writes `bm25_config.json` and the term-frequency statistics so Workstream 3 can reproduce this exact scoring policy. BM25 features do not use labels or original stored candidate position.


In [ ]:
N=num_docs; AVE_DL=total_doc_len/num_docs

@lru_cache(maxsize=300_000)
def idf(term):
    dfreq=doc_freq.get(term,0)
    return math.log1p((N-dfreq+0.5)/(dfreq+0.5))

@lru_cache(maxsize=100_000)
def query_term_set(s):
    return frozenset(tokenize(s))

def overlap(qset,tokens):
    if not qset: return 0.0
    return len(qset.intersection(tokens))/len(qset)

def bm25_score(query_tokens, name_tokens):
    if not query_tokens or not name_tokens: return 0.0
    tf=Counter(name_tokens)
    denom_const=BM25_K1 * (1-BM25_B + BM25_B*len(name_tokens)/AVE_DL)
    total=0.0
    for t in query_tokens:  # repeated query terms contribute repeatedly
        freq=tf.get(t,0)
        if freq:
            total += idf(t)*freq*(BM25_K1+1)/(freq+denom_const)
    return total

# Unit tests: overlap, duplicated tokens, BM25 absent matching token.
assert overlap(frozenset(['a','b','c']),['a','b','d']) == 2/3
assert overlap(frozenset(['a']),[]) == 0
assert bm25_score(('unseen-test-token',),('another-test-token',)) == 0.0
assert bm25_score(('a',),('a',)) > 0.0

output_cols=['session_id','candidate_pos','candidate_id','split','label','has_metadata',
             'query_name_overlap','query_brand_overlap','query_cate_overlap',
             'bm25_query_name','name_length','has_brand','shop_is_missing','cate_depth',
             'brand_product_count','shop_product_count','cate4_product_count']

writer=None; batch_count=0
try:
    for record_batch in tqdm(reader,desc='Scoring candidate rows'):
        part=record_batch.to_pandas()
        results=[]
        for r in part.itertuples(index=False):
            hasmeta=not pd.isna(r.matched_wid)
            if hasmeta:
                q=tokenize(r.query); qs=query_term_set(r.query)
                name=tokenize(r.name); brand=tokenize(r.brand_name)
                categories=set()
                for value in (r.cate_name_1,r.cate_name_2,r.cate_name_3,r.cate_name_4):
                    categories.update(tokenize(value))
                hasbrand=int(bool(brand))
                shop_missing=int(pd.isna(r.shop_id) or r.shop_id in ('','-1'))
                depth=3 if r.cate_name_3==r.cate_name_4 else 4
                results.append((int(r.session_id),int(r.candidate_pos),int(r.candidate_id),r.split,int(r.label),1,
                    overlap(qs,name), overlap(qs,brand) if hasbrand else np.nan,
                    overlap(qs,categories),bm25_score(q,name),len(name),hasbrand,shop_missing,depth,
                    brand_counts.get(r.brand_id,np.nan) if hasbrand else np.nan,
                    shop_counts.get(r.shop_id,np.nan) if not shop_missing else np.nan,
                    cate4_counts.get(r.cate_id_4,np.nan)))
            else:
                results.append((int(r.session_id),int(r.candidate_pos),int(r.candidate_id),r.split,int(r.label),0,
                    np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan))
        out=pd.DataFrame(results,columns=output_cols)
        # Enforce the same Arrow schema across every batch. Some batches have
        # only complete metadata (integer-like values), while later batches
        # contain missing metadata (NaN), which otherwise changes int -> float.
        integer_cols = ['session_id', 'candidate_pos', 'candidate_id', 'label', 'has_metadata']
        float_cols = [c for c in output_cols if c not in integer_cols + ['split']]
        out = out.astype({
            'session_id': 'int64', 'candidate_pos': 'int16',
            'candidate_id': 'int64', 'label': 'int8',
            'has_metadata': 'int8', 'split': 'string',
            **{c: 'float64' for c in float_cols}
        })
        table=pa.Table.from_pandas(out,preserve_index=False)
        if writer is None: writer=pq.ParquetWriter(FEATURE_PATH,table.schema,compression='zstd')
        writer.write_table(table)
        batch_count+=1
finally:
    if writer is not None: writer.close()
print('Feature batches:',batch_count)
print('Feature rows:',pq.read_metadata(FEATURE_PATH).num_rows)
assert pq.read_metadata(FEATURE_PATH).num_rows == pq.read_metadata(BASE_PATH).num_rows

Scoring candidate rows: 0it [00:00, ?it/s]

Feature batches: 384
Feature rows: 7665446


## 5. Validation and exploratory analysis

Before handing features to the modelling owners, the code checks that:

- Every input candidate is present exactly once after processing, including candidates without metadata.
- The split session counts and purchase totals match the sampled dataset.
- Overlap scores stay within `[0, 1]` and BM25 scores are finite and nonnegative where metadata exists.
- `(session_id, candidate_pos)` is unique within the combined candidate table.

The **training-only** table of average features by label is an exploratory diagnostic. It does **not** demonstrate better NDCG, purchase probability or online conversion. Actual ranking performance belongs to Workstreams 3 and 4.

Section 7 adds a small independent recalculation of actual rows. That checks formulas on selected examples, while the checks here examine the complete export.


In [ ]:
con2=duckdb.connect()
summary=con2.execute("""
SELECT split, count(*) AS candidate_rows, count(DISTINCT session_id) AS sessions,
       sum(CASE WHEN label=3 THEN 1 ELSE 0 END) AS purchases,
       avg(has_metadata) AS metadata_coverage,
       avg(bm25_query_name) AS avg_bm25
FROM read_parquet(?) GROUP BY split ORDER BY split
""",[str(FEATURE_PATH)]).df()
display(summary)

bad=con2.execute("""
SELECT count(*) FILTER (WHERE query_name_overlap<0 OR query_name_overlap>1 OR
                               query_brand_overlap<0 OR query_brand_overlap>1 OR
                               query_cate_overlap<0 OR query_cate_overlap>1) AS bad_overlap,
       count(*) FILTER (WHERE bm25_query_name<0 OR isnan(bm25_query_name) OR isinf(bm25_query_name)) AS bad_bm25,
       count(*) FILTER (WHERE has_metadata=0 AND bm25_query_name IS NOT NULL) AS bad_missing,
       count(*) AS total_rows,
       count(DISTINCT (session_id,candidate_pos)) AS unique_pairs
FROM read_parquet(?)
""",[str(FEATURE_PATH)]).fetchone()
print('bad_overlap, bad_bm25, bad_missing, rows, unique keys:',bad)
assert bad[0]==bad[1]==bad[2]==0 and bad[3]==bad[4]

if MAX_SESSIONS is None:
    assert summary.candidate_rows.sum()==7_665_446
    assert summary.purchases.sum()==10_428
    assert summary.sessions.sum()==85_864
    assert dict(zip(summary.split, summary.sessions))=={'train':60104,'val':12880,'test':12880}
print('All automated checks passed.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,candidate_rows,sessions,purchases,metadata_coverage,avg_bm25
0,test,1147673,12880,1562.0,0.995563,11.782041
1,train,5360982,60104,7293.0,0.995400,11.843145
2,val,1156791,12880,1573.0,0.995510,11.824260


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

bad_overlap, bad_bm25, bad_missing, rows, unique keys: (0, 0, 0, 7665446, 7665446)
All automated checks passed.


In [ ]:
# Training-only exploratory diagnostics. These are not model performance metrics.
feature_summary=con2.execute("""
SELECT label,count(*) AS n,
    avg(query_name_overlap) AS mean_name_overlap,
    avg(query_brand_overlap) AS mean_brand_overlap,
    avg(query_cate_overlap) AS mean_category_overlap,
    avg(bm25_query_name) AS mean_bm25
FROM read_parquet(?) WHERE split='train'
GROUP BY label ORDER BY label
""",[str(FEATURE_PATH)]).df()
display(feature_summary)
feature_summary.to_csv(DIAGNOSTICS_DIR/'train_feature_diagnostics.csv',index=False)

# Audit a small sample by hand, including unmatched metadata rows.
display(con2.execute('SELECT * FROM read_parquet(?) LIMIT 12',[str(FEATURE_PATH)]).df())

,label,n,mean_name_overlap,mean_brand_overlap,mean_category_overlap,mean_bm25
0,0,5248985,0.719247,0.093082,0.200733,11.818357
1,1,78395,0.719798,0.090054,0.206246,12.877801
2,2,26309,0.723037,0.104803,0.200022,13.286904
3,3,7293,0.746521,0.089664,0.235246,13.315408


,session_id,candidate_pos,candidate_id,split,label,has_metadata,query_name_overlap,query_brand_overlap,query_cate_overlap,bm25_query_name,name_length,has_brand,shop_is_missing,cate_depth,brand_product_count,shop_product_count,cate4_product_count
0,33418,32,84303384,train,0,1,0.00,0.0,0.00,0.000000,13.0,1.0,0.0,4.0,80.0,9145.0,8069.0
1,85422,56,33811859,test,0,1,0.00,0.0,0.00,0.000000,19.0,1.0,0.0,4.0,6.0,9145.0,9193.0
2,53870,84,84294794,train,0,1,0.50,0.0,0.00,6.339358,32.0,1.0,0.0,3.0,177.0,177.0,1475.0
3,44791,17,34508212,train,0,1,1.00,0.0,0.00,13.148674,22.0,1.0,0.0,4.0,476.0,124.0,2597.0
4,71294,67,32838799,val,0,1,0.00,0.0,0.00,0.000000,25.0,1.0,0.0,3.0,261.0,62.0,10590.0
5,72347,78,43985934,val,0,1,1.00,1.0,0.00,11.754262,21.0,1.0,0.0,3.0,335.0,93.0,3483.0
6,17633,23,19650441,train,0,1,1.00,0.0,0.00,7.344436,36.0,1.0,0.0,3.0,3641.0,1375.0,40667.0
7,78043,16,58357343,test,0,1,0.75,0.0,0.25,13.326760,18.0,1.0,0.0,3.0,5065.0,19.0,1018.0
8,46265,80,14001207,train,0,1,1.00,0.0,0.50,14.214059,18.0,1.0,0.0,3.0,128.0,94.0,57039.0
9,31902,70,11823164,train,0,1,0.00,0.0,0.00,0.000000,21.0,1.0,0.0,3.0,121.0,51.0,56026.0


## 6. Export the modelling datasets

The final feature table is split back into `train`, `val`, and `test` Parquet files without changing the original user-level split. Each row contains the 11 planned features, `has_metadata`, the target label, and identifiers needed for integration.

**What the modelling teammates should do:**

- **Workstream 3 (baselines):** sort the candidates *within each `session_id`* by `bm25_query_name` and calculate the team's agreed ranking metrics. Use the same BM25 configuration for every split; define how null scores and ties are handled.
- **Workstream 4 (LambdaMART):** join Workstream 1 history features, select predictor columns from the feature dictionary, and train the ranker using `label` as the supervised target and `session_id` as the ranking group.
- **Workstream 5 (attention):** these tabular features may be used alongside raw historical sequences if that model needs them.
- **Workstream 6 (diagnostics):** compare prediction files and perform cold-item / error analysis. Metadata availability alone is not evidence of cold-start improvement.

**Keys and leakage warning:** use `(session_id, candidate_pos)` to align each candidate row. Keep `candidate_id` to verify the product. Do **not** use `candidate_pos`, `session_id`, `candidate_id`, `split` or `label` as model predictors in the current setup. `candidate_pos` reflects a stored order that is label-informed. `label` is the prediction target.


In [ ]:
for split in ('train','val','test'):
    dst=RUN_DIR/f'query_item_{split}.parquet'
    # DuckDB COPY paths are SQL literals; escape embedded single quotes.
    src_sql = str(FEATURE_PATH).replace("'", "''")
    dst_sql = str(dst).replace("'", "''")
    con2.execute(f"COPY (SELECT * FROM read_parquet('{src_sql}') WHERE split='{split}') TO '{dst_sql}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(split, f'{dst.stat().st_size/1e6:.1f} MB', pq.read_metadata(dst).num_rows, 'rows')

feature_dict={
 'query_name_overlap':'unique query-token coverage in product name',
 'query_brand_overlap':'unique query-token coverage in brand name; null when brand missing',
 'query_cate_overlap':'unique query-token coverage in union of four category names',
 'bm25_query_name':'Okapi BM25 relevance vs product name using fixed full-catalogue IDF',
 'name_length':'product-name token count including duplicates',
 'has_brand':'1 when brand name exists',
 'shop_is_missing':'1 when shop_id absent/-1',
 'cate_depth':'3 when level-3 and level-4 category names are equal, else 4',
 'brand_product_count':'full catalogue products with same brand ID and present brand name',
 'shop_product_count':'full catalogue products with same non-missing shop ID',
 'cate4_product_count':'full catalogue products with same most-specific category ID',
 'has_metadata':'1 when candidate product is present in metadata'
}
pd.DataFrame([{'feature':k,'definition':v} for k,v in feature_dict.items()]).to_csv(
    RUN_DIR/'feature_dictionary.csv',index=False)
print('COMPLETE. Outputs:',RUN_DIR)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train 103.5 MB 5360982 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

val 23.0 MB 1156791 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test 22.7 MB 1147673 rows
COMPLETE. Outputs: /content/drive/MyDrive/JDsearch/workstream2_outputs/full


## Report note: AI-assisted development

The course requires an AI Interaction Log in **Section 4 (Appendix)** of the final report, recording essential prompts, AI responses, critical evaluation and revisions. A separate concise draft is provided with this notebook; it should be updated to reflect the checks personally carried out before submission.

It is important to distinguish AI-generated implementation suggestions from the actual project decisions, empirical runs, debugging and validation carried out by the group. The log should accurately disclose substantive assistance rather than implying the code was written entirely without it.


## 7. Independent feature spot-check (run after the full export)

**Goal:** Test mathematical correctness on a small set of *actual* JDSearch rows, including purchased products and missing-metadata cases. This is different from simply checking that numbers are within expected ranges.

The audit reads the saved candidate-pair table, candidate metadata, BM25 settings and catalogue summary tables. It independently recalculates overlaps, BM25 and catalogue attributes for selected records and compares each result with the exported feature value.

**Output:** `independent_feature_audit.csv`, with the checked examples and maximum absolute discrepancy.

**Scope:** A small deterministic spot-check, **not** an independent full-catalogue rescan or proof of predictive performance. It is designed to run after a Colab restart without rerunning Sections 1–6.

After reorganising the Drive folder, Section 7 reads from `full/intermediate/` and saves its audit to `full/diagnostics/`. It does not need the full catalogue scan to be repeated.


In [ ]:
# Optional path check: run this after Drive is mounted, without rerunning Sections 1-6.
from pathlib import Path
VERIFY_ROOT = Path('/content/drive/MyDrive/JDsearch/workstream2_outputs/full')
VERIFY_FILES = [
    VERIFY_ROOT / 'query_item_train.parquet',
    VERIFY_ROOT / 'query_item_val.parquet',
    VERIFY_ROOT / 'query_item_test.parquet',
    VERIFY_ROOT / 'bm25_config.json',
    VERIFY_ROOT / 'intermediate' / 'candidate_pairs.parquet',
    VERIFY_ROOT / 'intermediate' / 'matched_metadata.parquet',
    VERIFY_ROOT / 'intermediate' / 'query_item_all.parquet',
    VERIFY_ROOT / 'intermediate' / 'term_document_frequencies.parquet',
    VERIFY_ROOT / 'intermediate' / 'brand_catalogue_counts.parquet',
    VERIFY_ROOT / 'intermediate' / 'shop_catalogue_counts.parquet',
    VERIFY_ROOT / 'intermediate' / 'cate4_catalogue_counts.parquet',
]
missing = [str(p) for p in VERIFY_FILES if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
print('All required file paths found:', len(VERIFY_FILES))


In [ ]:
# Post-run audit: no need to rerun the 12.1M-product catalogue scan.
from pathlib import Path
from collections import Counter
import json, math
import pandas as pd
import duckdb
import pyarrow.parquet as pq

AUDIT_DIR = Path('/content/drive/MyDrive/JDsearch/workstream2_outputs/full')
AUDIT_INTERMEDIATE_DIR = AUDIT_DIR / 'intermediate'
AUDIT_DIAGNOSTICS_DIR = AUDIT_DIR / 'diagnostics'
AUDIT_DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
assert (AUDIT_INTERMEDIATE_DIR / 'query_item_all.parquet').exists(), 'Missing intermediate/query_item_all.parquet. Check the output folder.'

# Choose actual examples with metadata (including purchase, if available) and without metadata.
ac = duckdb.connect()
q = """SELECT f.session_id, f.candidate_pos, f.candidate_id, f.label,
              f.has_metadata, f.query_name_overlap, f.query_brand_overlap,
              f.query_cate_overlap, f.bm25_query_name, f.name_length,
              f.has_brand, f.shop_is_missing, f.cate_depth,
              f.brand_product_count, f.shop_product_count, f.cate4_product_count,
              b.query, m.name, m.brand_id, m.brand_name, m.shop_id,
              m.cate_name_1, m.cate_name_2, m.cate_name_3, m.cate_name_4,
              m.cate_id_4
       FROM read_parquet(?) f
       JOIN read_parquet(?) b USING (session_id, candidate_pos)
       LEFT JOIN read_parquet(?) m ON f.candidate_id = m.wid
       WHERE f.has_metadata = ? AND (? = -1 OR f.label = ?)
       LIMIT 5"""
paths = [str(AUDIT_INTERMEDIATE_DIR / 'query_item_all.parquet'),
         str(AUDIT_INTERMEDIATE_DIR / 'candidate_pairs.parquet'),
         str(AUDIT_INTERMEDIATE_DIR / 'matched_metadata.parquet')]
examples = pd.concat([
    ac.execute(q, paths + [1, 3, 3]).df(),
    ac.execute(q, paths + [1, 0, 0]).df(),
    ac.execute(q, paths + [0, -1, -1]).df()
], ignore_index=True).drop_duplicates(['session_id','candidate_pos'])

config = json.loads((AUDIT_DIR / 'bm25_config.json').read_text())
term_df = dict(ac.execute('SELECT term_id, df FROM read_parquet(?)',
                          [str(AUDIT_INTERMEDIATE_DIR / 'term_document_frequencies.parquet')]).fetchall())
brand_size = dict(ac.execute('SELECT brand_id, count FROM read_parquet(?)',
                            [str(AUDIT_INTERMEDIATE_DIR / 'brand_catalogue_counts.parquet')]).fetchall())
shop_size = dict(ac.execute('SELECT shop_id, count FROM read_parquet(?)',
                           [str(AUDIT_INTERMEDIATE_DIR / 'shop_catalogue_counts.parquet')]).fetchall())
category_size = dict(ac.execute('SELECT cate_id_4, count FROM read_parquet(?)',
                               [str(AUDIT_INTERMEDIATE_DIR / 'cate4_catalogue_counts.parquet')]).fetchall())

def audit_tokens(value):
    return [] if pd.isna(value) or value == '' else str(value).split('\x18')

def independent_overlap(query, document):
    unique_query = set(query)
    return len(unique_query.intersection(document)) / len(unique_query) if unique_query else 0.0

def independent_bm25(query, document):
    counts = Counter(document)
    k1, b = config['k1'], config['b']
    norm = k1 * (1 - b + b * len(document) / config['avgdl'])
    score = 0.0
    for term in query:  # repeated query terms count again, as documented
        tf = counts.get(term, 0)
        if tf:
            frequency = term_df.get(term, 0)
            idf = math.log(1 + (config['num_docs'] - frequency + 0.5) / (frequency + 0.5))
            score += idf * (tf * (k1 + 1) / (tf + norm))
    return score

checks = []
for r in examples.itertuples(index=False):
    query = audit_tokens(r.query)
    if r.has_metadata == 0:
        assert pd.isna(r.bm25_query_name) and pd.isna(r.query_name_overlap)
        checks.append({'session_id': r.session_id, 'candidate_id': r.candidate_id,
                       'case': 'no metadata', 'max_absolute_error': 0.0})
        continue
    name = audit_tokens(r.name)
    brand = audit_tokens(r.brand_name)
    category = set()
    for v in [r.cate_name_1, r.cate_name_2, r.cate_name_3, r.cate_name_4]:
        category.update(audit_tokens(v))
    expected = {
        'query_name_overlap': independent_overlap(query, name),
        'query_cate_overlap': independent_overlap(query, category),
        'bm25_query_name': independent_bm25(query, name),
        'name_length': len(name),
        'has_brand': int(bool(brand)),
        'shop_is_missing': int(pd.isna(r.shop_id) or r.shop_id in ('', '-1')),
        'cate_depth': 3 if r.cate_name_3 == r.cate_name_4 else 4,
        'cate4_product_count': category_size.get(r.cate_id_4, float('nan')),
    }
    expected['query_brand_overlap'] = independent_overlap(query, brand) if brand else float('nan')
    expected['brand_product_count'] = brand_size.get(r.brand_id, float('nan')) if brand else float('nan')
    expected['shop_product_count'] = shop_size.get(r.shop_id, float('nan')) if expected['shop_is_missing'] == 0 else float('nan')
    errors = []
    for key, target in expected.items():
        observed = getattr(r, key)
        if pd.isna(target):
            assert pd.isna(observed), (r.session_id, key, observed, target)
        else:
            assert math.isclose(float(observed), float(target), rel_tol=1e-9, abs_tol=1e-8), (r.session_id, key, observed, target)
            errors.append(abs(float(observed)-float(target)))
    checks.append({'session_id': r.session_id, 'candidate_id': r.candidate_id,
                   'case': 'metadata present', 'max_absolute_error': max(errors, default=0.0)})

audit_df = pd.DataFrame(checks)
audit_df.to_csv(AUDIT_DIAGNOSTICS_DIR / 'independent_feature_audit.csv', index=False)
print('Independent audit passed:', len(audit_df), 'real candidate rows; maximum error:',
      audit_df.max_absolute_error.max())
display(audit_df)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Independent audit passed: 15 real candidate rows; maximum error: 7.105427357601002e-15


,session_id,candidate_id,case,max_absolute_error
0,32513,70864072,metadata present,7.105427e-15
1,83229,29840983,metadata present,1.776357e-15
2,56066,36667384,metadata present,0.000000e+00
3,51550,19892660,metadata present,0.000000e+00
4,53638,90581020,metadata present,1.776357e-15
5,33418,84303384,metadata present,0.000000e+00
6,85422,33811859,metadata present,0.000000e+00
7,53870,84294794,metadata present,8.881784e-16
8,72800,34508212,metadata present,0.000000e+00
9,74697,32838799,metadata present,8.881784e-16


## 8. Model handoff checklist (run after Section 7)

This cell verifies the three final split files against expected row counts and writes `handoff_manifest.json`.

### What to share

| File | Who needs it | What is inside |
|---|---|---|
| `query_item_train.parquet` | Workstreams 3, 4 | Candidate rows and feature values for training |
| `query_item_val.parquet` | Workstreams 3, 4 | Validation data for model selection |
| `query_item_test.parquet` | Workstreams 3, 4, 6 | Held-out test data (final reporting only) |
| `bm25_config.json` | Workstream 3 | BM25 variant, `k1`, `b`, corpus size and scoring assumptions |
| `feature_dictionary.csv` | All model owners | Definitions of engineered features |
| `handoff_manifest.json` | All model owners | Row counts, join keys and predictor-exclusion rules |
| `independent_feature_audit.csv` | Reviewer/report | Evidence from manual-style recalculation of actual rows |

**Key rules:** Workstream 1 must use the *same original* `user_subset.parquet` row indices as `session_id`. Join candidate-level data by `(session_id, candidate_pos)` and check that `candidate_id` agrees. If Workstream 1 creates session-level features, join on `session_id` only. If it needs raw brand/shop/category identifiers, use the shared `matched_metadata.parquet` file; this metadata subset covers candidate products, not all products in histories.

**Do not treat** label, stored position, IDs, or the split as training predictors. This is a hard leakage/integration rule, not just a formatting convention.


In [ ]:
# Lightweight reproducibility and integration manifest.
from pathlib import Path
import json
import pyarrow.parquet as pq

HANDOFF_DIR = Path('/content/drive/MyDrive/JDsearch/workstream2_outputs/full')
output_names = ['query_item_train.parquet', 'query_item_val.parquet',
                'query_item_test.parquet']
row_counts = {name: pq.read_metadata(HANDOFF_DIR / name).num_rows for name in output_names}
assert row_counts == {
    'query_item_train.parquet': 5_360_982,
    'query_item_val.parquet': 1_156_791,
    'query_item_test.parquet': 1_147_673,
}, row_counts

feature_names = ['has_metadata', 'query_name_overlap', 'query_brand_overlap',
                 'query_cate_overlap', 'bm25_query_name', 'name_length',
                 'has_brand', 'shop_is_missing', 'cate_depth',
                 'brand_product_count', 'shop_product_count', 'cate4_product_count']
manifest = {
    'dataset': 'JDSearch sampled 85,864 sessions',
    'row_counts': row_counts,
    'join_keys': ['session_id', 'candidate_pos'],
    'verification_key': 'candidate_id',
    'label': 'label (0=none, 1=click, 2=cart, 3=purchase)',
    'model_features': feature_names,
    'forbidden_predictor_columns': ['session_id','candidate_pos','candidate_id','split','label'],
    'candidate_pos_note': 'Stored order is label-informed; key for alignment ONLY',
    'session_id_note': 'Original row index of user_subset.parquet; preserve original order',
    'bm25_config': 'bm25_config.json',
    'bm25_scores': 'bm25_query_name in each split Parquet file',
    'catalogue_assumption': 'Full static catalogue available at inference; not time-filtered',
    'missing_values': 'Use has_metadata and preserve feature nulls; imputations belong to the model pipeline',
    'cold_item_note': 'Metadata coverage is not cold-item performance. Derive cold status using train-only interaction sources.',
    'independent_audit': 'diagnostics/independent_feature_audit.csv',
    'intermediate_folder': 'intermediate/ (candidate pairs, matched product metadata and catalogue statistics)',
    'diagnostics_folder': 'diagnostics/ (validation and summary CSV/TXT files)',
    'files_to_share': output_names + ['bm25_config.json', 'feature_dictionary.csv',
                                      'handoff_manifest.json', 'diagnostics/independent_feature_audit.csv'],
}
(HANDOFF_DIR / 'handoff_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Handoff manifest saved:', HANDOFF_DIR / 'handoff_manifest.json')
print('All split rows:', sum(row_counts.values()))
print('Share the three split files, feature dictionary, BM25 config, audit, and manifest.')


Handoff manifest saved: /content/drive/MyDrive/JDsearch/workstream2_outputs/full/handoff_manifest.json
All split rows: 7665446
Share the three split files, feature dictionary, BM25 config, audit, and manifest.


## 9. Data summary for the report and handoff

This final section is **optional for feature computation but recommended for reporting**. It reads the existing exported files only and creates three short summary tables:

1. `handoff_dataset_summary.csv`: candidate rows, session counts, purchases and metadata coverage by split.
2. `handoff_feature_by_label_train.csv`: mean query-matching scores by interaction label, **training split only**.
3. `handoff_feature_missingness.csv`: null rates for the exported feature columns, grouped by split.

A text summary is also written to `workstream2_summary.txt`. This provides verified numbers to cite in the report rather than copying estimates from a previous smoke test.

**How to interpret:** Average BM25 scores by label show association, not ranking performance or purchase probability. The actual rank-based baseline (NDCG@K, etc.) is Workstream 3's responsibility. For cold-item claims, Workstream 6 must first define cold products from training interaction data.


In [ ]:
# Post-run summary: only reads existing split Parquet files; safe after a runtime restart.
from pathlib import Path
import json
import pandas as pd
import duckdb
import pyarrow.parquet as pq

SUMMARY_DIR = Path('/content/drive/MyDrive/JDsearch/workstream2_outputs/full')
SUMMARY_DIAGNOSTICS_DIR = SUMMARY_DIR / 'diagnostics'
SUMMARY_DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
paths = {s: SUMMARY_DIR / f'query_item_{s}.parquet' for s in ('train','val','test')}
assert all(p.exists() for p in paths.values()), 'Run Sections 1–6 first, or correct SUMMARY_DIR.'
db = duckdb.connect()
all_files = [str(paths[s]) for s in ('train','val','test')]

by_split = db.execute('''SELECT split, COUNT(*) AS candidate_rows,
  COUNT(DISTINCT session_id) AS sessions,
  SUM(CASE WHEN label = 3 THEN 1 ELSE 0 END) AS purchases,
  ROUND(100.0 * AVG(has_metadata), 3) AS metadata_coverage_pct
  FROM read_parquet(?) GROUP BY split ORDER BY split''', [all_files]).df()
assert int(by_split.candidate_rows.sum()) == 7_665_446
assert int(by_split.purchases.sum()) == 10_428
assert int(by_split.sessions.sum()) == 85_864
by_split.to_csv(SUMMARY_DIAGNOSTICS_DIR / 'handoff_dataset_summary.csv', index=False)

train_by_label = db.execute('''SELECT label, COUNT(*) AS candidate_rows,
  AVG(query_name_overlap) AS mean_name_overlap,
  AVG(query_brand_overlap) AS mean_brand_overlap,
  AVG(query_cate_overlap) AS mean_category_overlap,
  AVG(bm25_query_name) AS mean_bm25
  FROM read_parquet(?) GROUP BY label ORDER BY label''',
  [str(paths['train'])]).df()
train_by_label.to_csv(SUMMARY_DIAGNOSTICS_DIR / 'handoff_feature_by_label_train.csv', index=False)

feature_cols = ['has_metadata','query_name_overlap','query_brand_overlap',
 'query_cate_overlap','bm25_query_name','name_length','has_brand',
 'shop_is_missing','cate_depth','brand_product_count',
 'shop_product_count','cate4_product_count']
parts=[]
for split, path in paths.items():
    expr = ', '.join(f'ROUND(100.0 * AVG(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END), 3) AS "{c}"' for c in feature_cols)
    missing = db.execute(f'SELECT {expr} FROM read_parquet(?)', [str(path)]).df().iloc[0]
    parts.extend({'split':split,'feature':c,'null_pct':float(missing[c])} for c in feature_cols)
missingness = pd.DataFrame(parts)
missingness.to_csv(SUMMARY_DIAGNOSTICS_DIR / 'handoff_feature_missingness.csv',index=False)

bm25 = json.loads((SUMMARY_DIR / 'bm25_config.json').read_text())
lines = [
 'BT4222 / JDSearch — Workstream 2: Final data summary',
 'Full feature dataset: 85,864 search sessions; 7,665,446 candidate pairs; 10,428 purchases.',
 'Features: 4 query-product matching + 7 catalogue metadata + has_metadata indicator.',
 f"BM25: {bm25['variant']}; k1={bm25['k1']}; b={bm25['b']}; corpus={bm25['corpus']}",
 'Keys: (session_id, candidate_pos) for row matching; candidate_id for product verification.',
 'Do not train on candidate_pos, session_id, candidate_id, split, or label as predictors.',
 'Split summary:',
 by_split.to_string(index=False),
 'Train-only matching feature averages by interaction label:',
 train_by_label.to_string(index=False),
 'Interpretation: descriptive association only; evaluate ranking through Workstream 3.',
 'Cold-start effectiveness requires training-only cold-item definition and downstream evaluation.'
]
(SUMMARY_DIAGNOSTICS_DIR / 'workstream2_summary.txt').write_text('\n'.join(lines),encoding='utf-8')
print('DATASET SUMMARY'); display(by_split)
print('TRAIN-ONLY FEATURE SUMMARY'); display(train_by_label)
print('FEATURE NULL RATES'); display(missingness)
print('Summary files saved to',SUMMARY_DIAGNOSTICS_DIR)

DATASET SUMMARY


,split,candidate_rows,sessions,purchases,metadata_coverage_pct
0,test,1147673,12880,1562.0,99.556
1,train,5360982,60104,7293.0,99.540
2,val,1156791,12880,1573.0,99.551


TRAIN-ONLY FEATURE SUMMARY


,label,candidate_rows,mean_name_overlap,mean_brand_overlap,mean_category_overlap,mean_bm25
0,0,5248985,0.719247,0.093082,0.200733,11.818357
1,1,78395,0.719798,0.090054,0.206246,12.877801
2,2,26309,0.723037,0.104803,0.200022,13.286904
3,3,7293,0.746521,0.089664,0.235246,13.315408


FEATURE NULL RATES


,split,feature,null_pct
0,train,has_metadata,0.000
1,train,query_name_overlap,0.460
2,train,query_brand_overlap,5.752
3,train,query_cate_overlap,0.460
4,train,bm25_query_name,0.460
5,train,name_length,0.460
6,train,has_brand,0.460
7,train,shop_is_missing,0.460
8,train,cate_depth,0.460
9,train,brand_product_count,5.752


Summary files saved to /content/drive/MyDrive/JDsearch/workstream2_outputs/full


### What remains outside this notebook

- **Workstream 1:** Provide matching session/candidate keys and user-history features; confirm shared metadata conventions.
- **Workstream 3:** Evaluate BM25-only ranking with the same session groups, metric definitions, and treatment of null/tied scores as other baselines.
- **Workstream 4:** Ablate query/item features against history-only features in LambdaMART using validation sessions for tuning and test sessions for final reporting.
- **Workstream 6:** Define cold items from **training** interactions and evaluate subgroup ranking; do not infer cold-start gains from high metadata coverage.
- **All owners:** Retain successful notebook outputs in the submitted `.ipynb`, share accessible artifact links and document the shared split, join keys and feature-selection policy.
